## KNN

### KNN 建模步骤：

1. 数据准备 / Data Preparation —— 分离特征（X）和目标变量（y）
2. 类别变量编码 / Categorical Encoding —— 对 Color、Source 进行编码
3. 训练集与测试集划分 / Train-Test Split
4. 特征缩放 / Feature Scaling —— KNN基于距离计算，必须做标准化
5. 模型构建 / Model Building —— 初始化 KNeighborsClassifier
6. 模型训练 / Model Training
7. 模型预测 / Model Prediction
8. 模型评估 / Model Evaluation —— Accuracy、Precision、Recall、F1、ROC-AUC
9. 交叉验证 / Cross Validation —— 5-Fold CV
10. 超参数调优 / Hyperparameter Tuning —— 调整 n_neighbors、weights、p
11. Before vs After Tuning 对比
12. Confusion Matrix & ROC Curve

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/water_quality_cleaned.csv")

print("Number of observations:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

df.info()

Number of observations: 734756
Number of columns: 23

Column names:
['pH', 'Iron', 'Nitrate', 'Chloride', 'Lead', 'Zinc', 'Color', 'Turbidity', 'Fluoride', 'Copper', 'Odor', 'Sulfate', 'Conductivity', 'Chlorine', 'Manganese', 'Total Dissolved Solids', 'Source', 'Water Temperature', 'Air Temperature', 'Month', 'Day', 'Time of Day', 'Target']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 734756 entries, 0 to 734755
Data columns (total 23 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   pH                      734756 non-null  float64
 1   Iron                    734756 non-null  float64
 2   Nitrate                 734756 non-null  float64
 3   Chloride                734756 non-null  float64
 4   Lead                    734756 non-null  float64
 5   Zinc                    734756 non-null  float64
 6   Color                   734756 non-null  object 
 7   Turbidity               734756 non-null  float64
 8   F

In [3]:
print("Color unique values:", df['Color'].nunique())
print(df['Color'].unique())

print("\nSource unique values:", df['Source'].nunique())
print(df['Source'].unique())

Color unique values: 5
['Faint Yellow' 'Light Yellow' 'Colorless' 'Near Colorless' 'Yellow']

Source unique values: 8
['Lake' 'River' 'Spring' 'Ground' 'Stream' 'Reservoir' 'Aquifer' 'Well']


#### 1. Data preparation

In [4]:
X_knn = df.drop(columns=['Target'])
y_knn = df['Target']

print("X_knn shape:", X_knn.shape)
print("y_knn shape:", y_knn.shape)

X_knn shape: (734756, 22)
y_knn shape: (734756,)


#### 2. Categorical Encoding：Color, Source
* Source is unordered categorical variable, so we choose one-hot encoding.

* Color: Colorless<Near Colorless<Faint Yellow<Light Yellow<Yellow, so we decide to use ordinal encoding

In [5]:
# Color: ordinal encoding (domain-based order)
color_order = ['Colorless', 'Near Colorless', 'Faint Yellow', 'Light Yellow', 'Yellow']
color_mapping = {cat: i for i, cat in enumerate(color_order)}
X_knn['Color'] = X_knn['Color'].map(color_mapping)

# Source: one-hot encoding (nominal variable)
X_knn = pd.get_dummies(X_knn, columns=['Source'], drop_first=False)

print("X_knn shape after encoding:", X_knn.shape)
print(X_knn.columns.tolist())

X_knn shape after encoding: (734756, 29)
['pH', 'Iron', 'Nitrate', 'Chloride', 'Lead', 'Zinc', 'Color', 'Turbidity', 'Fluoride', 'Copper', 'Odor', 'Sulfate', 'Conductivity', 'Chlorine', 'Manganese', 'Total Dissolved Solids', 'Water Temperature', 'Air Temperature', 'Month', 'Day', 'Time of Day', 'Source_Aquifer', 'Source_Ground', 'Source_Lake', 'Source_Reservoir', 'Source_River', 'Source_Spring', 'Source_Stream', 'Source_Well']


#### 3. Train-Test Split

In [6]:
from sklearn.model_selection import train_test_split

X_train_knn, X_test_knn, y_train_knn, y_test_knn = train_test_split(
    X_knn, y_knn,
    test_size=0.2,
    random_state=42,
    stratify=y_knn
)

print("X_train_knn shape:", X_train_knn.shape)
print("X_test_knn shape:", X_test_knn.shape)
print("y_train_knn shape:", y_train_knn.shape)
print("y_test_knn shape:", y_test_knn.shape)

X_train_knn shape: (587804, 29)
X_test_knn shape: (146952, 29)
y_train_knn shape: (587804,)
y_test_knn shape: (146952,)


---
## From here on: KNN-specific steps

Everything above follows the same data preparation as the Decision Tree notebook
(encoding + train-test split). Decision Tree does not need feature scaling because
splits are based on thresholds, not distance. **KNN is different** — it classifies a
point based on the distance to its nearest neighbors, so features with larger numeric
ranges (e.g. Total Dissolved Solids) would dominate the distance calculation over
features with small ranges (e.g. pH) unless we scale everything first.

#### 4. Feature Scaling

We use `StandardScaler` (z-score standardization) instead of MinMaxScaler because the
EDA stage showed several features (e.g. Lead, Iron) contain outliers. MinMaxScaler
would compress most of the data into a narrow range if a single extreme value sets the max.

The scaler is fit **only on the training set**, then applied to the test set, to avoid
leaking test-set statistics into training.

In [7]:
from sklearn.preprocessing import StandardScaler

scaler_knn = StandardScaler()

X_train_knn_scaled = scaler_knn.fit_transform(X_train_knn)
X_test_knn_scaled  = scaler_knn.transform(X_test_knn)

# Convert back to DataFrame for readability
X_train_knn_scaled = pd.DataFrame(X_train_knn_scaled, columns=X_train_knn.columns, index=X_train_knn.index)
X_test_knn_scaled  = pd.DataFrame(X_test_knn_scaled,  columns=X_test_knn.columns,  index=X_test_knn.index)

print("Mean after scaling (train, should be ~0):")
print(X_train_knn_scaled.mean().round(2).head())
print("\nStd after scaling (train, should be ~1):")
print(X_train_knn_scaled.std().round(2).head())

Mean after scaling (train, should be ~0):
pH         -0.0
Iron       -0.0
Nitrate    -0.0
Chloride   -0.0
Lead        0.0
dtype: float64

Std after scaling (train, should be ~1):
pH          1.0
Iron        1.0
Nitrate     1.0
Chloride    1.0
Lead        1.0
dtype: float64


#### 5. Model Building: KNeighborsClassifier

Starting with a default K=5 baseline before any tuning.

In [8]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)

print(knn_model)

KNeighborsClassifier()


#### 6. Model Training

In [9]:
knn_model.fit(X_train_knn_scaled, y_train_knn)

print("Model training complete")

Model training complete


#### 7. Model Prediction

In [10]:
y_pred_knn = knn_model.predict(X_test_knn_scaled)

print("Prediction complete")
print("First 10 prediction results:", y_pred_knn[:10])

Prediction complete
First 10 prediction results: [1 1 0 1 0 0 0 0 0 0]


#### 8. Model Evaluation: Accuracy、Precision、Recall、F1、ROC-AUC

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_pred_proba_knn = knn_model.predict_proba(X_test_knn_scaled)[:, 1]

accuracy_knn  = accuracy_score(y_test_knn, y_pred_knn)
precision_knn = precision_score(y_test_knn, y_pred_knn)
recall_knn    = recall_score(y_test_knn, y_pred_knn)
f1_knn        = f1_score(y_test_knn, y_pred_knn)
roc_auc_knn   = roc_auc_score(y_test_knn, y_pred_proba_knn)

print("Accuracy:", accuracy_knn)
print("Precision:", precision_knn)
print("Recall:", recall_knn)
print("F1 Score:", f1_knn)
print("ROC-AUC:", roc_auc_knn)

Accuracy: 0.816035167945996
Precision: 0.6519035476222431
Recall: 0.43080561714708054
F1 Score: 0.5187795934351526
ROC-AUC: 0.8265588362054561


#### 9. Cross Validation: 5-Fold CV

In [12]:
from sklearn.model_selection import cross_val_score

cv_scores_knn = cross_val_score(knn_model, X_train_knn_scaled, y_train_knn, cv=5, scoring='accuracy')

print("5-Fold CV Accuracy scores:", cv_scores_knn)
print("Mean CV Accuracy:", cv_scores_knn.mean())
print("Std CV Accuracy:", cv_scores_knn.std())

5-Fold CV Accuracy scores: [0.8128546  0.81506622 0.81458137 0.81424112 0.81433311]
Mean CV Accuracy: 0.8142152828456826
Std CV Accuracy: 0.0007381208310688677
